# 06b · 가우스 과정과 이차 변동 (Gaussian Processes and Quadratic Variation)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Ross 12e §10.7 (가우스 과정) · Shreve II §3.3.2 (BM 의 결합정규 분포), §3.4 (일차·이차 변동) · G&S 4e §9.6 (가우스 과정), §13.2 (BM 의 성질) · Lawler 2e §8.1 · Øksendal §2.2 (Kolmogorov 확장·연속성) · Karlin & Taylor 2e §7.2 |
| 선수 노트북 | 06a (BM 의 정의·공분산·미분불가능), 00c (다변량 정규·Cholesky), 00b (MC 오차) |
| 예상 소요 | 90분 |
| 상태 | draft |

06a 에서 BM 을 "독립 정규 증분을 쌓은 것"으로 만들었다. 이 노트북은 같은 대상을 두 번째 얼굴로 본다: **BM 은 평균 0, 공분산 $\min(s,t)$ 인 가우스 과정(Gaussian process)** 이고, 그 공분산 행렬을 Cholesky 로 풀면 놀랍게도 "증분을 그냥 더하라"가 나온다 — 같은 난수를 쓰면 두 구성이 소수점까지 같다. 후반부는 경로의 **거칠음을 숫자로** 잰다: 이차 변동(quadratic variation) $\sum(\Delta B)^2\to T$ 와 총변동(total variation) $\sum|\Delta B|\to\infty$. 매끄러운 함수는 정반대($0$ 과 유한)라서, 이 두 숫자가 바로 07a–07b 에서 보통 미적분이 깨지고 이토 미적분이 필요한 이유다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import norm
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.brownian import brownian_paths, coarsen, quadratic_variation
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("06b")
setup_plots()
N_PATHS = scale(20_000)      # 공분산 검사용 경로 수; FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_PATHS_QV = scale(2_000)    # 이차·총변동 분포용 경로 수 (경로당 1000 걸음)
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_PATHS_QV={N_PATHS_QV}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (06a) 표준 BM 의 정의 네 조건은?

<details><summary>정답</summary>

$B_0=0$; 독립 증분; $B_t-B_s\sim N(0,t-s)$ ($s<t$); 연속 경로.

</details>

**Q2.** (06a) $\mathrm{Cov}(B_s,B_t)$ 와 $\rho(B_1,B_4)$ 는?

<details><summary>정답</summary>

$\mathrm{Cov}(B_s,B_t)=\min(s,t)$ (겹치는 과거의 길이). $\rho(B_1,B_4)=\min(1,4)/\sqrt{1\cdot4}=\sqrt{1/4}=0.5$.

</details>

**Q3.** (06a) 차분몫 $(B_{t+h}-B_t)/h$ 의 분포와 $h\to0$ 에서의 운명은?

<details><summary>정답</summary>

$N(0,1/h)$. 표준편차 $h^{-1/2}\to\infty$ 이므로 극한이 없다 — 경로는 (거의 모든 곳에서) 미분 불가능하다. 이 노트북의 총변동 $=\infty$ 는 같은 사실의 다른 얼굴이다.

</details>

**Q4.** (05d) 폴리아 항아리의 빨강 비율이 수렴하는 이유와 극한 분포는?

<details><summary>정답</summary>

비율은 $[0,1]$ 에 갇힌 유계 마팅게일이므로 마팅게일 수렴정리에 의해 a.s. 수렴한다. 극한은 $\mathrm{Beta}(a,b)$ (초기 빨강 $a$, 파랑 $b$). 이 노트북의 이차 변동도 부분열을 따라 a.s. 수렴하지만, 근거는 Chebyshev + Borel–Cantelli 다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **가우스 과정**(Gaussian process): 임의의 유한 시점 $(t_1,\dots,t_k)$ 에 대해 벡터 $(X_{t_1},\dots,X_{t_k})$ 가 다변량 정규인 과정. **평균함수** $m(t)=E[X_t]$ 와 **공분산 커널**(covariance kernel) $K(s,t)=\mathrm{Cov}(X_s,X_t)$ 가 모든 유한차원 분포를 결정하고, Kolmogorov 확장정리로 그런 과정이 존재한다 (Øksendal §2.2). 커널은 대칭·양반정치(positive semidefinite)여야 한다 — 유한 격자에서 $K_{ij}=K(t_i,t_j)$ 가 공분산 행렬이어야 하므로.
- **BM 의 가우스 과정 표현**: $m\equiv0$, $K(s,t)=\min(s,t)$. 격자 $0<t_1<\dots<t_m$ 에서 $K_{ij}=\min(t_i,t_j)$, $L=\mathrm{chol}(K)$ ($K=LL^\top$, $L$ 하삼각), $Z\sim N(0,I_m)$ 이면 $B=LZ$ 가 격자 위 BM 의 표본이다 (00c 의 Cholesky 표본추출).
- **분할과 변동**: 분할 $\pi=\{0=t_0<t_1<\dots<t_n=T\}$, 메시 $|\pi|=\max_i\Delta t_i$ 에 대해
  $$Q_\pi(T)=\sum_{i=0}^{n-1}(B_{t_{i+1}}-B_{t_i})^2\quad(\text{이차 변동}),\qquad V_\pi(T)=\sum_{i=0}^{n-1}|B_{t_{i+1}}-B_{t_i}|\quad(\text{총변동, 일차 변동}).$$
  등간격 $t_i=iT/n$ 이면 $Q_n(T)$, $V_n(T)$ 로 쓴다. 함수 $f$ 가 **유계변동**(bounded variation)이라는 것은 $\sup_\pi V_\pi<\infty$ 라는 뜻이다.
- **비교용 커널** (4.2 에서 그림으로 본다): 브라운 다리 $K(s,t)=\min(s,t)-st/T$ (06e 예고; $t=T$ 에서 분산 0 이라 특이), 제곱지수(squared exponential) $K(s,t)=\exp(-(s-t)^2/2\ell^2)$ (무한 번 미분가능한 매끄러운 경로), 랜덤 기울기 $K(s,t)=st$ ($X_t=tZ$, E1·E3).

### 2.2 정리 1 — 이차 변동의 $L^2$ 수렴 (Shreve II §3.4.2, G&S 4e §13.2)

등간격 분할 $t_i=iT/n$ 에 대해
$$E[Q_n(T)]=T,\qquad \mathrm{Var}(Q_n(T))=\frac{2T^2}{n}.$$
따라서 $Q_n(T)\to T$ in $L^2$ (그리고 확률수렴), 그리고 부분열 $n=2^k$ 를 따라 a.s. 수렴한다.

가정이 왜 필요한가:
- **증분 독립**: 분산이 각 항의 분산의 합으로 갈라진다 (교차항 0). 독립이 아니면 $\mathrm{Var}(Q_n)$ 에 $n^2$ 개의 교차항이 남아 $1/n$ 으로 사라질 이유가 없다.
- **증분 $N(0,\Delta t)$**: $E[\Delta B^4]=3\Delta t^2$ 이 필요하다 — 4차 모멘트가 $\Delta t^2$ 차수여야 분산이 $1/n$ 으로 사라진다. 무한분산 증분이면 실패한다 (E3(b)).
- **메시 $\to0$**: 고정 $n$ 에서 $Q_n$ 은 표준편차 $T\sqrt{2/n}$ 의 확률변수이지 상수가 아니다. "이차 변동 $=T$" 는 극한 명제다.

<details><summary>증명</summary>

$\Delta_i=B_{t_{i+1}}-B_{t_i}$ 는 독립이고 $\Delta_i\sim N(0,\Delta t)$, $\Delta t=T/n$.

**평균.** $E[\Delta_i^2]=\Delta t$ 이므로 $E[Q_n]=n\Delta t=T$.

**분산.** 정규분포의 4차 모멘트 $E[\Delta_i^4]=3(\Delta t)^2$ 에서
$$\mathrm{Var}(\Delta_i^2)=E[\Delta_i^4]-(E[\Delta_i^2])^2=3\Delta t^2-\Delta t^2=2\Delta t^2 .$$
독립성으로 $\mathrm{Var}(Q_n)=\sum_i\mathrm{Var}(\Delta_i^2)=2n\Delta t^2=2T^2/n\to0$. 즉 $E[(Q_n-T)^2]=\mathrm{Var}(Q_n)\to0$: $L^2$ 수렴.

**a.s. 부분열.** Chebyshev: $P(|Q_n-T|>\epsilon)\le\mathrm{Var}(Q_n)/\epsilon^2=2T^2/(n\epsilon^2)$. $n=2^k$ 를 넣으면 $\sum_k2T^2/(2^k\epsilon^2)=4T^2/\epsilon^2<\infty$ 이므로 Borel–Cantelli 로 $|Q_{2^k}-T|>\epsilon$ 이 유한 번만 일어난다. $\epsilon=1/m$ 으로 가산 합집합을 취하면 $Q_{2^k}(T)\to T$ a.s. (Shreve II §3.4.2 는 등간격이 아닌 일반 분할에서도 $|\pi|\to0$ 이면 $L^2$ 수렴함을 같은 계산으로 보인다: $\mathrm{Var}(Q_\pi)=2\sum\Delta t_i^2\le2|\pi|T$.) $\square$

</details>

### 2.3 정리 2 — 총변동은 무한 (Shreve II §3.4.1, G&S 4e §13.2)

부분열 $n=2^k$ 에 대해 $V_n(T)\to\infty$ a.s. 따라서 BM 의 경로는 (거의 확실히) 어떤 구간 $[0,T]$ 에서도 유계변동이 아니다. 또한 등간격 분할에서
$$E[V_n(T)]=\sqrt{\frac{2nT}{\pi}},\qquad\mathrm{Var}(V_n(T))=T\Big(1-\frac2\pi\Big)\ \ (n\text{ 에 무관}).$$

가정이 왜 필요한가:
- **경로 연속성**: $\max_i|\Delta_i|\to0$ 을 주는 유일한 근거다 — 연속함수는 컴팩트 $[0,T]$ 에서 균등연속이다. 점프가 있는 과정(푸아송)은 $\max|\Delta_i|$ 가 0 으로 가지 않으므로 이 논법이 통하지 않는다.
- **$Q_n\to T>0$ (정리 1)**: 0 이 아닌 극한이 있어야 부등식이 힘을 쓴다. 매끄러운 함수는 $Q_n\to0$ 이라 $V_n\ge Q_n/\max|\Delta_i|$ 가 $0/0$ 꼴이 되어 아무 결론도 없다 — 실제로 유계변동이다.

<details><summary>증명</summary>

**무한.** 각 항에서 $\Delta_i^2\le(\max_j|\Delta_j|)\,|\Delta_i|$ 이므로 더하면
$$Q_n=\sum_i\Delta_i^2\le\Big(\max_i|\Delta_i|\Big)\sum_i|\Delta_i|=\Big(\max_i|\Delta_i|\Big)V_n .$$
부분열 $n=2^k$ 에서 좌변 $\to T>0$ a.s. (정리 1). 경로가 연속이므로 $[0,T]$ 에서 균등연속이고, 메시 $T/n\to0$ 이므로 $\max_i|\Delta_i|\to0$ a.s. 따라서 $V_n\ge Q_n/\max_i|\Delta_i|\to T/0^+=\infty$ a.s. 분할을 세분하면 $V_\pi$ 는 커지기만 하므로 $\sup_\pi V_\pi=\infty$: 유계변동이 아니다.

**평균과 분산.** $|\Delta_i|=\sqrt{\Delta t}\,|Z|$, $Z\sim N(0,1)$ 이고 $E|Z|=\sqrt{2/\pi}$, $\mathrm{Var}|Z|=1-2/\pi$ 이므로
$$E[V_n]=n\sqrt{\Delta t}\sqrt{2/\pi}=n\sqrt{\frac{2T}{\pi n}}=\sqrt{\frac{2nT}{\pi}},\qquad \mathrm{Var}(V_n)=n\Delta t\Big(1-\frac2\pi\Big)=T\Big(1-\frac2\pi\Big).$$
평균은 $\sqrt n$ 으로 자라는데 표준편차는 상수다 — 그래서 $V_n$ 은 그림에서 거의 결정론적인 곡선 $\sqrt{2nT/\pi}$ 위에 올라탄다. $\square$

</details>

### 2.4 정리 3 — 가우스 과정 특성화 (Ross 12e §10.7, G&S 4e §9.6)

연속 경로를 가진 과정 $X$ 가 표준 BM $\iff$ $X$ 는 평균 0, 공분산 $\min(s,t)$ 인 가우스 과정.

가정이 왜 필요한가:
- **가우스성**: '무상관 → 독립' 으로 넘어가는 유일한 다리다. 공분산이 $\min(s,t)$ 이지만 가우스가 아닌 과정 — 예: 보상 푸아송 과정 $N_t-t$ 는 $\mathrm{Cov}=\min(s,t)$ 이지만 점프한다 — 는 BM 이 아니다.
- **연속 경로**: 유한차원 분포만으로는 경로 성질(연속·유계변동 여부)이 결정되지 않는다. Kolmogorov 연속성 정리($E|B_t-B_s|^4=3|t-s|^2$)가 연속 버전의 존재를 준다.

*증명 스케치.* (⇒) 06a 정리 1: 독립 정규 증분의 합은 결합정규이고 $\mathrm{Cov}(B_s,B_t)=\mathrm{Var}(B_s)=s$ ($s\le t$). (⇐) 증분 $X_t-X_s$ 는 가우스 벡터의 선형결합이라 정규이고 분산은 $K(t,t)-2K(s,t)+K(s,s)=t-2s+s=t-s$. 겹치지 않는 증분 $X_b-X_a$, $X_d-X_c$ ($a<b\le c<d$) 의 공분산은 $K(b,d)-K(b,c)-K(a,d)+K(a,c)=b-b-a+a=0$; 결합정규이므로 무상관 = 독립. $X_0\sim N(0,K(0,0)=0)$ 이므로 $X_0=0$. 존재성·연속성은 인용 (Øksendal §2.2, G&S 4e §13.2). $\square$

### 2.5 직관

공분산 커널은 "어느 시각들이 얼마나 함께 움직이는가"의 표다. $\min(s,t)$ 는 **공유하는 과거의 길이만큼만** 상관한다는 뜻이고, 이것을 Cholesky 로 풀면 '증분을 그냥 더하라'가 나온다: $L_{ij}=\sqrt{\Delta t}$ ($j\le i$) 인 하삼각행렬은 곧 누적합 연산자이고, $(LL^\top)_{ij}=\Delta t\cdot\#\{k\le\min(i,j)\}=\min(t_i,t_j)$. 격자 위 BM 은 두 방법이 같은 $Z$ 를 쓰면 소수점까지 같다.

이차 변동은 "시간 $t$ 동안 경로가 흔들린 에너지"다. BM 은 정확히 $t$, 매끄러운 함수는 0 을 갖는다 (증분이 $\Delta t$ 차수라 제곱하면 $\Delta t^2$, $n$ 개 더해도 $\to0$). 그래서 $f(B_t)$ 의 테일러 전개에서 2차항 $\frac12f''(B_t)(\Delta B)^2\approx\frac12f''(B_t)\Delta t$ 가 살아남고(07b 이토 공식), $\sum|\Delta B|=\infty$ 라서 $\int f\,dB$ 를 경로별 리만–스틸체스 적분으로 정의할 수 없다(07a). 4.5 의 망원 항등식 $B_T^2=2\sum B_{t_i}\Delta B_i+\sum(\Delta B_i)^2$ 이 이 이야기의 가장 작은 버전이다.

### 2.6 함정 (traps)

- Cholesky 격자에 $t=0$ 을 넣으면 $K$ 의 0 행·0 열 때문에 특이행렬이 되어 실패한다. 격자는 $t_1>0$ 부터 잡고, $B_0=0$ 은 따로 붙인다.
- 제곱지수 커널은 수치적으로 거의 특이(조건수 $\sim10^{18}$): 대각에 지터(jitter) $10^{-8}$ 을 더해야 Cholesky 가 된다. 이론이 아니라 부동소수점 문제다. 브라운 다리 커널도 $t=T$ 에서 분산 0 이라 격자에 $T$ 가 들어가면 같은 처방이 필요하다.
- "이차 변동 $=t$" 는 분할을 무한히 잘게 한 극한. 고정 $n$ 에서는 표준편차 $t\sqrt{2/n}$ 의 확률변수다 ($t=1$, $n=1000$: 0.045).
- 총변동이 무한이라고 경로가 불연속인 것은 아니다. 연속이면서 유계변동이 아닌 함수다 (반대로 점프가 있어도 유계변동일 수 있다: 푸아송 경로).
- $\mathrm{Var}(Q_n)=2T^2/n$ 은 $T^2$ 에 비례: $T$ 가 크면 같은 $n$ 에서 상대오차는 같지만 절대오차는 커진다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — 특히 "이차 변동이 $t$" 가 왜 "경로가 거칠다" 와 같은 말인지, 그리고 왜 매끄러운 함수는 이차 변동이 0 인지 두 문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 아래 값을 먼저 적어 보세요. 손계산 힌트: $\min(s,t)$, $T\sqrt{2/n}$, $\sqrt{2nT/\pi}$, 그리고 $\sin$ 의 한 주기 동안 오르내린 거리.

In [ ]:
predictions = {
    # E[B_{0.5} B_2] 는?
    "cov_B_half_B2": None,
    # T=1, 등간격 n=1000 에서 이차 변동 Q_n 의 표준편차는?
    "sd_qv_n1000": None,
    # T=1, n=1000 에서 총변동의 평균 E[V_n] 은? (n=16384 이면 얼마인지도 생각해 보세요)
    "mean_tv_n1000": None,
    # 매끄러운 함수 f(t)=sin(2 pi t), t in [0,1] 의 n=1000 이차 변동은? (총변동은?)
    "qv_sin_n1000": None,
    # 격자 t_k = 0.05k (k=1..40) 위 K_ij = min(t_i,t_j) 의 Cholesky 인자 L 의 하삼각 성분 L_ij (j<=i) 값은?
    "chol_entry": None,
}

## 4. Simulate — 시뮬레이션

먼저 이 노트북에서만 쓰는 작은 도우미를 정의한다. `gp_paths` 는 임의의 커널로 격자 위 가우스 과정을 Cholesky 표본추출하고(격자에 $t=0$ 을 넣지 말 것 — 함정 1), `total_variation` 은 spkit 의 `quadratic_variation` 의 일차 버전이다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def gp_paths(t, kernel_fn, rng, n_paths, jitter=0.0):
    """격자 t (0 제외) 위 평균 0 가우스 과정의 Cholesky 표본. 행이 경로. jitter 는 대각에 더하는 수치 안정화 항."""
    K = kernel_fn(t[:, None], t[None, :]) + jitter * np.eye(len(t))
    L = np.linalg.cholesky(K)
    return rng.standard_normal((n_paths, len(t))) @ L.T

def total_variation(X):
    """격자 위 총변동 sum |X_{k+1} - X_k| (axis 1), 경로별 스칼라."""
    return np.abs(np.diff(X, axis=-1)).sum(axis=-1)

k_bm = lambda s, t: np.minimum(s, t)                                  # BM
k_bridge = lambda T_: (lambda s, t: np.minimum(s, t) - s * t / T_)   # 브라운 다리 [0, T_]
k_se = lambda ell: (lambda s, t: np.exp(-(s - t) ** 2 / (2 * ell ** 2)))   # 제곱지수 (매끄러움)
k_slope = lambda s, t: s * t                                          # 랜덤 기울기 X_t = t Z (E3)

### 4.1 증분 구성의 공분산 검사 (Fig 1)

`brownian_paths` 는 06a 의 증분 누적합 구성이다. 21 격자점($\Delta t=0.1$, $T=2$) 위에서 표본 공분산 행렬 $\hat C=B^\top B/N$ 을 $\min(t_i,t_j)$ 와 나란히 놓는다. 한 성분 $E[B_{0.5}B_2]$ 은 `mc_estimate` 로 SE 까지 잰다 ($\mathrm{Var}(B_{0.5}B_2)=1.25$ 이므로 SE $\approx\sqrt{1.25/N}$).

In [ ]:
t1, B1 = brownian_paths(N_PATHS, 20, 2.0, rng)          # (N_PATHS, 21), dt = 0.1
C_hat = B1.T @ B1 / N_PATHS                              # 표본 공분산 (평균 0 이므로 중심화 불필요)
C_exact = np.minimum.outer(t1, t1)                       # min(t_i, t_j)
max_err = np.abs(C_hat - C_exact).max()
cov_inc_hat = mc_estimate(B1[:, 5] * B1[:, 20])          # t1[5] = 0.5, t1[20] = 2.0
print(f"grid points {t1.size}, dt = {t1[1]:.2f};  max |C_hat - min(s,t)| = {max_err:.4f}")
print(f"E[B_0.5 B_2] (increments) = {cov_inc_hat}   (exact min(0.5, 2) = 0.5)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
for ax, C, title in zip(axes, (C_hat, C_exact), (f"empirical  B^T B / N  (N = {N_PATHS})", "exact  min(s, t)")):
    im = ax.imshow(C, origin="lower", extent=(0, 2, 0, 2), vmin=0, vmax=2, cmap="viridis")
    ax.set_xlabel("t"); ax.set_ylabel("s"); ax.set_title(title); ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.85, label="Cov(B_s, B_t)")
fig.suptitle(f"Fig 1. Covariance of Brownian motion on a grid   (max abs error = {max_err:.3f})")
plt.show()

두 열지도가 같은 계단 모양이다: 대각선 위 $s$, 아래 $t$ — 즉 $\min(s,t)$. 최대 오차는 $N$ 이 10배 커지면 $\sqrt{10}$ 배 줄어드는 MC 잡음이다.

### 4.2 Cholesky 표본추출과 '누적합' 발견 (Fig 2)

이제 반대 방향: 커널 $\min(t_i,t_j)$ 만 주고 00c 의 Cholesky 표본추출 $B=LZ$ 로 BM 을 만든다. 격자는 $t_k=0.05k$, $k=1,\dots,40$ ($t=0$ 제외, $t=0.5$ 는 인덱스 9, $t=2$ 는 인덱스 39). 먼저 $L$ 자체를 들여다본다 — 예측 `chol_entry` 를 확인하세요.

In [ ]:
DT, M = 0.05, 40
tg = np.arange(1, M + 1) * DT                            # 0.05, 0.10, ..., 2.00
K = np.minimum.outer(tg, tg)
L = np.linalg.cholesky(K)                                # K = L L^T
L_cumsum = np.sqrt(DT) * np.tril(np.ones((M, M)))       # sqrt(dt) 로 채운 하삼각 = 누적합 연산자
same_L = np.allclose(L, L_cumsum)
print("L[:4, :4] =\n", L[:4, :4].round(6))
print(f"chol(min) == sqrt(dt)*tril(ones): {same_L}   (max abs diff {np.abs(L - L_cumsum).max():.1e}, sqrt(dt) = {np.sqrt(DT):.6f})")

Z = rng.standard_normal((N_PATHS, M))                    # 같은 난수로 두 구성을 비교
X_chol = Z @ L.T                                         # 행별로 L Z
X_inc = np.cumsum(np.sqrt(DT) * Z, axis=1)               # 증분 누적합
same_paths = np.allclose(X_chol, X_inc)
i_half, i_two = 9, 39                                    # tg[9] = 0.5, tg[39] = 2.0
cov_chol_hat = mc_estimate(X_chol[:, i_half] * X_chol[:, i_two])
var2_chol_hat = mc_estimate(X_chol[:, i_two] ** 2)
print(f"Cholesky paths == cumsum paths (same Z): {same_paths}   (max abs diff {np.abs(X_chol - X_inc).max():.1e})")
print(f"E[B_0.5 B_2] (Cholesky) = {cov_chol_hat}   (exact 0.5)")
print(f"E[B_2^2]     (Cholesky) = {var2_chol_hat}   (exact 2)")

$L$ 의 하삼각이 전부 $\sqrt{\Delta t}$ 다. 즉 "$\min$ 커널의 Cholesky" = "증분을 쌓는 연산" 이고, 같은 $Z$ 를 쓰면 두 경로는 분포만 같은 게 아니라 **값이** 같다 (trap 카드). 같은 `gp_paths` 에 다른 커널을 넣으면 전혀 다른 거칠음의 경로가 나온다.

In [ ]:
JITTER, ELL = 1e-8, 0.3
K_se = k_se(ELL)(tg[:, None], tg[None, :])
print(f"cond(K_se) = {np.linalg.cond(K_se):.1e}  ->  with jitter {JITTER:g}: {np.linalg.cond(K_se + JITTER * np.eye(M)):.1e}")
panels = [("kernel min(s,t)  (BM)", gp_paths(tg, k_bm, rng, 5)),
          ("min(s,t) - st/T  (Brownian bridge, T=2)", gp_paths(tg, k_bridge(2.0), rng, 5, jitter=JITTER)),
          (f"exp(-(s-t)^2/2l^2), l={ELL}  (smooth)", gp_paths(tg, k_se(ELL), rng, 5, jitter=JITTER))]
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, (title, P) in zip(axes, panels):
    plot_paths(tg, P, ax=ax, n_show=5, lw=1.2)
    ax.set_title(title, fontsize=10); ax.axhline(0, color="k", lw=0.6)
axes[0].set_ylabel("X_t")
fig.suptitle("Fig 2. Five Cholesky-sampled paths per covariance kernel (same sampler, different kernel)")
plt.tight_layout(); plt.show()

브라운 다리는 $t=2$ 에서 0 으로 묶이고, 제곱지수 커널의 경로는 미분가능해 보인다 — 실제로 무한 번 미분가능하다. 거칠음은 커널의 대각선 근처 모양이 결정한다: $\min(s,t)$ 는 대각선에서 꺾여 있고(도함수 불연속), $\exp(-(s-t)^2/2\ell^2)$ 는 매끄럽다.

### 4.3 한 경로의 이차·총변동을 `coarsen` 으로 본다 (Fig 3)

$n=16384$ 격자 위 경로 **하나**를 만들고, `coarsen` 으로 같은 경로를 $2^k$ 배 성긴 격자($n_k=16384/2^k$, $k=0,\dots,10$)에서 다시 본다. 같은 격자에서 매끄러운 함수 $f(t)=\sin2\pi t$ 도 평가한다. 정리 1·2 의 예측: $Q_n\to1$, $V_n\approx\sqrt{2n/\pi}$; $f$ 는 $Q_n^f\approx2\pi^2/n\to0$, $V_n^f\to4$.

In [ ]:
N_FINE = 16384
t3, B3 = brownian_paths(1, N_FINE, 1.0, rng)
n_k, Q_k, V_k, Qf_k, Vf_k = [], [], [], [], []
for k in range(11):
    tk, Xk = coarsen(t3, B3, 2 ** k)
    fk = np.sin(2 * np.pi * tk)[None, :]                 # 같은 격자 위 매끄러운 함수
    n_k.append(tk.size - 1)
    Q_k.append(quadratic_variation(Xk)[0, -1]); V_k.append(total_variation(Xk)[0])
    Qf_k.append(quadratic_variation(fk)[0, -1]); Vf_k.append(total_variation(fk)[0])
n_k, Q_k, V_k, Qf_k, Vf_k = map(np.array, (n_k, Q_k, V_k, Qf_k, Vf_k))
print("      n      Q_n(BM)   V_n(BM)  sqrt(2n/pi)   Q_n(sin)   V_n(sin)")
for row in zip(n_k, Q_k, V_k, np.sqrt(2 * n_k / np.pi), Qf_k, Vf_k):
    print("{:7d}   {:8.4f}  {:8.3f}   {:8.3f}    {:8.5f}   {:7.4f}".format(*row))

In [ ]:
fig, (axA, axB, axC) = plt.subplots(1, 3, figsize=(14, 4))
axA.semilogx(n_k, Q_k, "o-", label="Brownian path  $Q_n$")
axA.semilogx(n_k, Qf_k, "s--", color="C2", label="sin(2 pi t)  $Q_n^f$")
axA.axhline(1.0, color="k", ls="--", lw=1, label="T = 1")
axA.set_xlabel("n (log)"); axA.set_ylabel("quadratic variation on [0, 1]"); axA.legend(fontsize=8)
axA.set_title("Fig 3a. $Q_n$ of one path vs n")
axB.plot(t3, quadratic_variation(B3)[0], color="C0", lw=1.2, label=f"running $Q(t)$, n = {N_FINE}")
axB.plot(t3, t3, "k--", lw=1, label="t")
axB.set_xlabel("t"); axB.set_ylabel("$\\sum_{t_i \\leq t} (\\Delta B_i)^2$"); axB.legend(fontsize=8)
axB.set_title("Fig 3b. Running quadratic variation = t")
axC.loglog(n_k, V_k, "o-", label="Brownian path  $V_n$")
axC.loglog(n_k, np.sqrt(2 * n_k / np.pi), "k--", lw=1, label="sqrt(2n/pi), slope 1/2")
axC.loglog(n_k, Vf_k, "s--", color="C2", label="sin(2 pi t)  $V_n^f$ -> 4")
axC.set_xlabel("n (log)"); axC.set_ylabel("total variation on [0, 1] (log)"); axC.legend(fontsize=8)
axC.set_title("Fig 3c. $V_n$ grows like sqrt(n)")
plt.tight_layout(); plt.show()

(a) 같은 경로를 성기게 볼수록 $Q_n$ 이 $1$ 주위에서 더 크게 흔들린다 — 표준편차 $\sqrt{2/n}$. 매끄러운 $f$ 는 기울기 $-1$ 로 0 을 향한다. (b) 누적 이차 변동이 대각선 $t$ 에 붙어 있다: "$dB^2=dt$". (c) 총변동은 $\sqrt n$ 으로 자라 무한으로 가고, $f$ 는 4 에 고정된다 (한 주기 동안 $0\to1\to-1\to0$ 의 이동 거리). 경로 하나라 SE 가 없다 — 분포는 다음 셀에서 잰다.

### 4.4 여러 경로에서 $Q_n$, $V_n$ 의 분포 (Fig 4)

$n=1000$, $T=1$ 경로 `N_PATHS_QV` 개. 정리 1: $Q_n\approx N(1,\,2/1000)$ (독립 $\chi^2$ 항 1000 개의 합이라 CLT 로 거의 정규). 정리 2: $E[V_n]=\sqrt{2000/\pi}$, $\mathrm{sd}(V_n)=\sqrt{1-2/\pi}$.

In [ ]:
N_QV, T_QV = 1000, 1.0
t4, B4 = brownian_paths(N_PATHS_QV, N_QV, T_QV, rng)     # (N_PATHS_QV, 1001)
Q4 = quadratic_variation(B4)[:, -1]                      # 경로별 Q_1000(1)
V4 = total_variation(B4)                                 # 경로별 V_1000(1)
qv_mean_hat = mc_estimate(Q4)
qv_var_hat = mc_estimate((Q4 - T_QV) ** 2)               # E[(Q_n - T)^2] = Var(Q_n)
tv_mean_hat = mc_estimate(V4)
sd_Q_hat = Q4.std(ddof=1)
sd_Q_se = sd_Q_hat / np.sqrt(2 * (N_PATHS_QV - 1))       # 정규 표본의 sd 의 근사 SE
print(f"E[Q_1000]        = {qv_mean_hat}   (exact T = {T_QV})")
print(f"E[(Q_1000-1)^2]  = {qv_var_hat:.5f}   (exact 2T^2/n = {2 * T_QV**2 / N_QV})")
print(f"sd(Q_1000)       = {sd_Q_hat:.4f} ± {sd_Q_se:.4f}   (exact T sqrt(2/n) = {T_QV * np.sqrt(2 / N_QV):.4f})")
print(f"E[V_1000]        = {tv_mean_hat}   (exact sqrt(2nT/pi) = {np.sqrt(2 * N_QV * T_QV / np.pi):.4f});  sd(V) = {V4.std(ddof=1):.3f} (exact {np.sqrt(T_QV * (1 - 2 / np.pi)):.3f})")

In [ ]:
xg = np.linspace(0.8, 1.2, 300)
fig, ax = plt.subplots()
plot_hist_vs_pdf(Q4, xg, norm.pdf(xg, loc=T_QV, scale=T_QV * np.sqrt(2 / N_QV)), ax=ax, bins=30,
                 label_pdf="N(1, 2/1000) pdf")
ax.set_xlabel("$Q_{1000}(1)$ per path"); ax.set_ylabel("density")
ax.set_title(f"Fig 4. Quadratic variation over {N_PATHS_QV} paths, n = 1000, T = 1")
ax.text(0.03, 0.95, f"sample sd = {sd_Q_hat:.4f}\nsqrt(2/n) = {np.sqrt(2 / N_QV):.4f}",
        transform=ax.transAxes, va="top", fontsize=9, bbox=dict(boxstyle="round", fc="white", alpha=0.8))
plt.show()

고정 $n$ 에서 $Q_n$ 은 상수가 아니라 폭 $\sqrt{2/n}$ 의 종 모양이다(함정 3). $n$ 을 100배 키우면 폭이 10배 줄어든다 — 그 극한이 "$=T$".

### 4.5 왜 보통 미적분이 깨지는가 — 망원 항등식 (Fig 5)

같은 경로들에 대해 $B_1^2=\sum_i(B_{t_{i+1}}^2-B_{t_i}^2)$ 를 전개하면 정확한 대수 항등식
$$B_T^2=\underbrace{2\sum_iB_{t_i}\Delta B_i}_{2I_n}+\underbrace{\sum_i(\Delta B_i)^2}_{Q_n}$$
이 나온다. $I_n$ 은 좌단점 리만합(07a 의 이토 적분 $\int_0^1B\,dB$ 의 이산판)이고, 마팅게일 변환이라 평균 0 이다. 그러면 $E[B_1^2]=1$ **전부**가 둘째 항 $Q_n\to1$ 에서 와야 한다. 보통 미적분($\int_0^1 2x\,dx=1$ 로 $B_1^2=2\int B\,dB$)에는 이 항이 없다.

In [ ]:
dB = np.diff(B4, axis=1)                                 # (N_PATHS_QV, 1000)
I4 = np.sum(B4[:, :-1] * dB, axis=1)                     # 좌단점 리만합 sum B_{t_i} dB_i
lhs, rhs = B4[:, -1] ** 2, 2 * I4 + Q4
identity_ok = np.allclose(lhs, rhs)
riemann_hat = mc_estimate(I4)
print(f"B_1^2 == 2*I + Q for every path: {identity_ok}   (max abs diff {np.abs(lhs - rhs).max():.1e})")
print(f"E[sum B dB] = {riemann_hat}   (exact 0;  sd ~ sqrt(T^2/2) = {np.sqrt(0.5):.3f})")
print(f"E[Q_1000]   = {qv_mean_hat}   (exact 1)   ->  E[B_1^2] = 2*0 + 1")

In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(11, 4))
axL.scatter(lhs, rhs, s=6, alpha=0.5)
lim = lhs.max() * 1.05
axL.plot([0, lim], [0, lim], "k--", lw=1, label="y = x")
axL.set_xlabel("$B_1^2$"); axL.set_ylabel("$2\\sum B_{t_i}\\Delta B_i + \\sum(\\Delta B_i)^2$"); axL.legend()
axL.set_title("Fig 5a. Telescoping identity, path by path")
axR.hist(I4, bins=30, density=True, alpha=0.5, label=f"$I_n = \\sum B\\,\\Delta B$  (mean {riemann_hat.mean:+.3f})")
axR.hist(Q4, bins=30, density=True, alpha=0.5, label=f"$Q_n = \\sum(\\Delta B)^2$  (mean {qv_mean_hat.mean:.3f})")
axR.axvline(0, color="C0", ls="--", lw=1); axR.axvline(1, color="C1", ls="--", lw=1)
axR.set_xlabel("value per path"); axR.set_ylabel("density"); axR.legend(fontsize=8)
axR.set_title("Fig 5b. The two terms: mean 0 and mean 1")
plt.tight_layout(); plt.show()

왼쪽: 점이 전부 대각선 위 — 항등식은 확률이 아니라 대수다. 오른쪽: $I_n$ 은 0 을 중심으로 넓게(사실 $\frac12(B_1^2-1)$ 의 분포), $Q_n$ 은 1 에 뾰족하게. 극한에서 $d(B^2)=2B\,dB+dt$ — 07b 이토 공식의 씨앗이다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드에서 계산한다: $\min(s,t)$, $T\sqrt{2/n}$, $2T^2/n$, $\sqrt{2nT/\pi}$, $\sqrt{\Delta t}$, $2n\sin^2(\pi/n)$ (등간격 격자에서 $\sin2\pi t$ 의 이차 변동은 $\sum(\Delta f)^2=\sum4\sin^2(\pi/n)\cos^2(2\pi t_i+\pi/n)$ 의 합이 정확히 $2n\sin^2(\pi/n)$).

In [ ]:
f4 = np.sin(2 * np.pi * t4)[None, :]                     # n = 1000 격자 위 매끄러운 함수
qv_sin_direct = quadratic_variation(f4)[0, -1]
tv_sin_direct = total_variation(f4)[0]
exact = {
    "cov": min(0.5, 2.0),                                  # min(s, t)
    "var2": 2.0,                                           # Var(B_2) = 2
    "chol": np.sqrt(DT),                                   # sqrt(dt)
    "qv_mean": T_QV,                                       # E[Q_n] = T
    "sd_qv": T_QV * np.sqrt(2 / N_QV),                     # sd(Q_n) = T sqrt(2/n)
    "qv_var": 2 * T_QV**2 / N_QV,                          # Var(Q_n) = 2T^2/n
    "tv_mean": np.sqrt(2 * N_QV * T_QV / np.pi),           # E[V_n] = sqrt(2nT/pi)
    "qv_sin": 2 * N_QV * np.sin(np.pi / N_QV) ** 2,        # sum (Delta sin)^2 on the uniform grid
    "riemann": 0.0,                                        # martingale transform
}
rows = [
    {"name": "E[B_0.5 B_2] (increments, 4.1)", "predicted": predictions["cov_B_half_B2"], "estimate": cov_inc_hat, "exact": exact["cov"]},
    {"name": "E[B_0.5 B_2] (Cholesky, 4.2)", "predicted": predictions["cov_B_half_B2"], "estimate": cov_chol_hat, "exact": exact["cov"]},
    {"name": "E[B_2^2] (Cholesky, 4.2)", "predicted": None, "estimate": var2_chol_hat, "exact": exact["var2"]},
    {"name": "Cholesky entry L_ij, j<=i (exact identity)", "predicted": predictions["chol_entry"], "estimate": float(L[20, 3]), "exact": exact["chol"]},
    {"name": "E[Q_1000(1)]", "predicted": None, "estimate": qv_mean_hat, "exact": exact["qv_mean"]},
    {"name": "sd(Q_1000(1))", "predicted": predictions["sd_qv_n1000"], "estimate": float(sd_Q_hat), "se": float(sd_Q_se), "exact": exact["sd_qv"]},
    {"name": "E[(Q_1000 - 1)^2] = Var(Q_1000)", "predicted": None, "estimate": qv_var_hat, "exact": exact["qv_var"]},
    {"name": "E[V_1000(1)]", "predicted": predictions["mean_tv_n1000"], "estimate": tv_mean_hat, "exact": exact["tv_mean"]},
    {"name": "Q_1000 of sin(2 pi t) (deterministic)", "predicted": predictions["qv_sin_n1000"], "estimate": float(qv_sin_direct), "exact": exact["qv_sin"]},
    {"name": "E[sum B_{t_i} dB_i] (left Riemann sum, 4.5)", "predicted": None, "estimate": riemann_hat, "exact": exact["riemann"]},
]
print(f"V_1000 of sin(2 pi t) = {tv_sin_direct:.6f} (exact 4);  E[V_16384] reference = {np.sqrt(2 * 16384 / np.pi):.2f}")
Markdown(compare_table(rows))

각 행이 어긋난다면 무엇이 의심되는가:
- **E[B_0.5 B_2] (increments)**: `brownian_paths` 의 증분 분산이 $\Delta t$ 가 아니거나 인덱스 5·20 이 $t=0.5,2$ 가 아니다.
- **E[B_0.5 B_2] (Cholesky), E[B_2^2]**: `K = min(t_i, t_j)` 를 잘못 만들었거나 `Z @ L.T` 대신 `Z @ L` 을 썼다 (상삼각으로 쌓으면 미래를 더한다).
- **Cholesky entry**: 격자에 $t=0$ 이 들어갔거나 간격이 균등하지 않다 — 균등하지 않으면 $L_{ij}=\sqrt{\Delta t_j}$ 로 열마다 달라진다.
- **E[Q_1000]**: 증분 분산이 $\Delta t$ 와 다르다 (예: $T$ 를 잊고 `normal(0, 1/n)` 처럼 sd 자리에 분산을 넣음).
- **sd(Q), Var(Q)**: 증분이 독립이 아니거나 4차 모멘트가 $3\Delta t^2$ 이 아니다 — 정리 1 의 두 가정 (E3(b) 가 이 행을 일부러 깨뜨린다).
- **E[V_1000]**: $E|Z|=\sqrt{2/\pi}$ 를 $1$ 로 착각했거나 `np.abs` 없이 더했다.
- **Q of sin**: 격자가 $[0,1]$ 전체(한 주기)가 아니다. 어긋난다고 BM 이론이 틀린 것은 아니다 — 순수 결정론 검산.
- **E[sum B dB]**: 우단점 $B_{t_{i+1}}\Delta B_i$ 를 썼다면 평균이 $+1$ 로 나온다 (이토 vs 다른 적분 규약, 07a).

In [ ]:
assert same_L and same_paths and identity_ok             # 정확 항등식 (MC 아님): chol = sqrt(dt) tril, LZ = cumsum, B^2 = 2I + Q
assert_close(cov_inc_hat, exact["cov"], k=4, name="E[B_0.5 B_2] increments")
assert_close(cov_chol_hat, exact["cov"], k=4, name="E[B_0.5 B_2] Cholesky")
assert_close(var2_chol_hat, exact["var2"], k=4, name="E[B_2^2] Cholesky")
assert_close(qv_mean_hat, exact["qv_mean"], k=4, name="E[Q_1000]")
assert_close(qv_var_hat, exact["qv_var"], k=4, name="E[(Q_1000-1)^2]")
assert_close(tv_mean_hat, exact["tv_mean"], k=4, name="E[V_1000]")
assert_close(riemann_hat, exact["riemann"], k=4, name="E[sum B dB]")
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1 계산

손으로: (a) $\mathrm{Cov}(B_1-B_{0.5},\,B_2)$; (b) $\mathrm{Corr}(B_1,B_2)$; (c) 브라운 다리 커널 $K(s,t)=\min(s,t)-st$ ($T=1$) 에서 $\mathrm{Var}(X_{1/2})$; (d) $K(s,t)=st$ 도 유효한(양반정치) 공분산 커널이다. 이 가우스 과정의 경로는 어떻게 생겼는가? 왜 BM 이 아닌가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 쌍선형성: $\mathrm{Cov}(B_1,B_2)-\mathrm{Cov}(B_{0.5},B_2)=\min(1,2)-\min(0.5,2)=1-0.5=0.5$. (독립 증분으로 봐도 같다: $B_1-B_{0.5}$ 는 $B_2-B_{0.5}$ 의 일부이고 $B_{0.5}$ 와 독립이므로 $\mathrm{Cov}=\mathrm{Var}(B_1-B_{0.5})=0.5$.)

(b) $\min(1,2)/\sqrt{1\cdot2}=1/\sqrt2=0.7071$.

(c) $K(\tfrac12,\tfrac12)=\tfrac12-\tfrac14=\tfrac14$ — 다리는 양 끝이 묶여 있어 가운데 분산이 BM 의 $\tfrac12$ 보다 작다.

(d) $K(s,t)=st=\phi(s)\phi(t)$ ($\phi(t)=t$) 꼴은 계수 1 의 양반정치 커널이다: $\sum_{ij}c_ic_jt_it_j=(\sum c_it_i)^2\ge0$. 실현은 $X_t=tZ$, $Z\sim N(0,1)$ — 원점을 지나는 **랜덤 기울기의 직선**. 증분 $X_t-X_s=(t-s)Z$ 는 서로 독립이 아니고(모두 같은 $Z$) 분산이 $(t-s)^2$ 이라 $\Delta t$ 에 비례하지 않는다. 경로는 매끄럽고 이차 변동은 0 이다 (E3(a)).

```python
print(min(1, 2) - min(0.5, 2), 1 / np.sqrt(2), 0.5 - 0.25)       # 0.5, 0.7071, 0.25
t_e1, B_e1 = brownian_paths(20_000, 4, 2.0, rng)                 # dt = 0.5: t = 0, .5, 1, 1.5, 2
print(mc_estimate((B_e1[:, 2] - B_e1[:, 1]) * B_e1[:, 4]))        # ≈ 0.5
print(np.corrcoef(B_e1[:, 2], B_e1[:, 4])[0, 1])                  # ≈ 0.7071
```

</details>

### E2 유도 후 시뮬레이션 검증

$E[V_n(T)]=\sqrt{2nT/\pi}$ 와 $\mathrm{Var}(Q_n(T))=2T^2/n$ 을 일반 $T$ 에 대해 유도하라 (본문은 $T=1$ 로 확인했다). 그런 다음 `brownian_paths(2000, 300, 3.0, rng)` 로 $T=3$, $n=300$ 에서 `mc_estimate(V)` 와 `mc_estimate((Q - 3)**2)` 를 구해 `assert_close` 로 검증하라. 두 추정량의 SE 를 미리 계산해 보라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$\Delta t=T/n$. $|\Delta_i|=\sqrt{\Delta t}|Z|$ 에서 $E[V_n]=n\sqrt{\Delta t}\sqrt{2/\pi}=\sqrt{2nT/\pi}$, $\mathrm{Var}(V_n)=n\Delta t(1-2/\pi)=T(1-2/\pi)$. $\Delta_i^2$ 의 분산은 $2\Delta t^2$ 이므로 $\mathrm{Var}(Q_n)=2n(T/n)^2=2T^2/n$.

$T=3$, $n=300$: $E[V]=\sqrt{1800/\pi}=23.9365$, $\mathrm{sd}(V)=\sqrt{3(1-2/\pi)}=1.044$ → SE $=1.044/\sqrt{2000}=0.023$. $\mathrm{Var}(Q)=18/300=0.06$; $(Q-3)^2\approx0.06\cdot\chi^2_1$ 이라 sd $\approx\sqrt2\cdot0.06=0.085$ → SE $\approx0.0019$. 시뮬레이션은 두 값 모두 4 SE 안에 들어온다.

```python
T_e2, n_e2 = 3.0, 300
t_e2, B_e2 = brownian_paths(2000, n_e2, T_e2, rng)
Q_e2 = quadratic_variation(B_e2)[:, -1]; V_e2 = total_variation(B_e2)
ev_exact, vq_exact = np.sqrt(2 * n_e2 * T_e2 / np.pi), 2 * T_e2**2 / n_e2
est_V, est_VQ = mc_estimate(V_e2), mc_estimate((Q_e2 - T_e2) ** 2)
print(est_V, ev_exact)            # ≈ 23.92 ± 0.023  vs 23.9365
print(est_VQ, vq_exact)           # ≈ 0.062 ± 0.002  vs 0.06
assert_close(est_V, ev_exact, k=4, name="E[V_300(3)]")
assert_close(est_VQ, vq_exact, k=4, name="Var(Q_300(3))")
```

</details>

### E3 가정을 깨보기

(a) 커널 $K(s,t)=st$ 로 `gp_paths` 를 돌려 $n=1000$ 격자($t_k=k/1000$, $T=1$) 에서 $Q_n$ 과 $V_n$ 을 구하라. (힌트: 이 커널의 격자 행렬은 계수 1 이라 Cholesky 에 지터가 필요하다 — 왜? 지터가 결과에 얼마나 영향을 주는지도 생각해 보라.) 정리 1 의 어느 가정이 깨지는가?
(b) 증분을 $N(0,\Delta t)$ 대신 분산이 무한한 $\sqrt{\Delta t}\cdot t_2$ (자유도 2 스튜던트 $t$, `rng.standard_t(2, size)`) 로 바꿔 $Q_n$ 의 경로별 히스토그램을 그려라. $n$ 을 100, 1000, 10000 으로 키워도 $Q_n$ 이 안정되는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $K=tt^\top$ 는 계수 1 이라 격자 행렬이 특이하다(고유값 하나만 양수); `cholesky` 는 양정치가 아니면 실패하므로 지터 $\epsilon$ 을 더한다. 지터는 각 격자점에 독립 $N(0,\epsilon)$ 잡음을 붙이는 것과 같아서 $Q_n$ 에 $\approx2n\epsilon$, $V_n$ 에 $\approx n\sqrt{2\epsilon}\cdot\sqrt{2/\pi}$ 를 더한다 — $\epsilon=10^{-12}$ 이면 각각 $2\times10^{-9}$, $10^{-3}$ 으로 무시할 만하다 ($10^{-8}$ 이면 $V_n$ 에 0.11 이 더해져 눈에 띈다).
결과: $X_t=tZ$ 이므로 $\Delta X_i=Z\Delta t$, $Q_n=Z^2T^2/n$ → $E[Q_n]=T^2/n=0.001\to0$, $V_n=|Z|T$ (유한, 평균 $\sqrt{2/\pi}=0.798$). 깨진 가정: **증분 독립**과 **증분 분산 $\propto\Delta t$** 둘 다 (증분은 모두 같은 $Z$ 의 배수이고 분산은 $\Delta t^2$). 경로가 매끄러우니 이차 변동이 0 인 것은 $\sin$ 과 같은 이유다.

(b) $t_2$ 는 분산이 무한이라 $E[\Delta^2]=\infty$, $E[\Delta^4]=\infty$: 정리 1 의 평균·분산 계산이 둘 다 무의미하다. 히스토그램은 오른쪽 꼬리가 극단적으로 길고 몇 경로가 평균을 지배한다. $n$ 을 키워도 폭이 줄지 않으며 중앙값은 오히려 $\log n$ 처럼 천천히 커진다 ($\sum t_i^2$ 은 지수 1 의 안정분포 영역이라 $n\log n$ 으로 정규화된다). 즉 $Q_n$ 은 어떤 상수로도 수렴하지 않는다: 이차 변동이 $t$ 인 것은 "증분이 작다"가 아니라 **정규 증분의 4차 모멘트가 $3\Delta t^2$** 이기 때문이다.

```python
# (a)
tg3 = np.arange(1, 1001) / 1000
X3 = gp_paths(tg3, k_slope, rng, 500, jitter=1e-12)
X3 = np.concatenate([np.zeros((500, 1)), X3], axis=1)           # X_0 = 0 붙이기
Q3, V3, Z3 = quadratic_variation(X3)[:, -1], total_variation(X3), X3[:, -1]
print(mc_estimate(Q3), 1 / 1000)                                 # ≈ 0.0010 ± 0.0001 vs 0.001
print(mc_estimate(V3), np.sqrt(2 / np.pi), np.abs(V3 - np.abs(Z3)).max())   # ≈ 0.80, |V - |Z|| ~ 1e-4
# (b)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for ax, n in zip(axes, (100, 1000, 10_000)):
    dX = np.sqrt(1 / n) * rng.standard_t(2, (2000, n))
    Qt = (dX ** 2).sum(axis=1)
    ax.hist(np.clip(Qt, 0, 40), bins=40, density=True)
    ax.set_title(f"n = {n}: median {np.median(Qt):.1f}, max {Qt.max():.0f}"); ax.set_xlabel("Q_n (clipped at 40)")
plt.tight_layout(); plt.show()
# 중앙값 ≈ 5, 7, 10 으로 커지고 최댓값은 수천: 수렴 없음
```

</details>

## 7. 정리

1. BM 은 평균 0, 공분산 $\min(s,t)$ 인 (연속) 가우스 과정이며, 격자에서 Cholesky 로 뽑은 표본은 증분 누적합과 값까지 같다: $\mathrm{chol}(\min(t_i,t_j))=\sqrt{\Delta t}\cdot\mathrm{tril}(\mathbf 1)$.
2. 이차 변동 $Q_n(T)\to T$: 평균 $T$, 분산 $2T^2/n$ ($L^2$ 수렴, 부분열 $2^k$ 에서 a.s.). 고정 $n$ 에서는 폭 $T\sqrt{2/n}$ 의 확률변수.
3. 총변동 $V_n\to\infty$ (평균 $\sqrt{2nT/\pi}$, 분산 $T(1-2/\pi)$ 상수): 경로는 연속이지만 유계변동이 아니다. 근거는 $Q_n\le\max|\Delta_i|\cdot V_n$ 과 균등연속.
4. 매끄러운 함수는 정반대: 이차 변동 $\to0$ ($\sin2\pi t$, $n=1000$: $0.0197\approx2\pi^2/n$), 총변동 유한 (4). 랜덤 기울기 $tZ$ 도 마찬가지.
5. 망원 항등식 $B_T^2=2\sum B_{t_i}\Delta B_i+\sum(\Delta B_i)^2$ 에서 둘째 항이 $T$ 로 살아남는다 — $dB^2=dt$, $d(B^2)=2B\,dB+dt$: 이토 공식의 씨앗.

**Trap 카드**
- Q: Cholesky 로 만든 격자 BM 과 증분 누적합 BM 은 '분포가 같다' 인가 '값이 같다' 인가?
- A: 같은 $Z$ 를 쓰면 값까지 같다. $\min(t_i,t_j)$ 의 Cholesky 인자는 $\sqrt{\Delta t}$ 로 채운 하삼각행렬이고 $LZ$ 가 곧 누적합이다. (격자가 등간격이 아니면 열 $j$ 가 $\sqrt{\Delta t_j}$ — 여전히 누적합.)

**다음 노트북: 06c** — 거칠음의 결과 하나: 격자점 사이에서 경로가 어디까지 올라갔는지 모른다. 06c 는 반사 원리로 최댓값·도달시간의 분포를 구하고, 격자 시뮬레이션의 편향($\propto\sqrt{\Delta t}$)을 재서 브라운 다리 보정으로 없앤다.

log/progress.md 한 줄 템플릿:
`- [ ] 06b 가우스 과정과 이차 변동 — YYYY-MM-DD, 예측 5개 중 __개 적중, 막힌 곳: ____, 다시 볼 것: 정리 2 증명의 부등식 Q_n <= max|Δ| · V_n`